In [1]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd

In [2]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [3]:
# Load chart 1 data as df
copper_df = pd.read_excel('Chartbook_Peru_Critical Minerals_LMC&TC_v1.xlsx', sheet_name='Chart 1', usecols=[0,1,2], skiprows=1)
copper_df.columns = ['year', 'year_month', 'price']
copper_df['date'] = pd.to_datetime(
    copper_df['year_month'].str.replace('M', '-'),
    format='%Y-%m'
)
copper_df

,year,year_month,price,date
0,1960,1960M01,0.324319,1960-01-01
1,1960,1960M02,0.330216,1960-02-01
2,1960,1960M03,0.310711,1960-03-01
3,1960,1960M04,0.327948,1960-04-01
4,1960,1960M05,0.310711,1960-05-01
...,...,...,...,...
793,2026,2026M02,5.874482,2026-02-01
794,2026,2026M03,5.683066,2026-03-01
795,2026,2026M04,5.874482,2026-04-01
796,2026,2026M05,6.143009,2026-05-01


In [4]:
# Create chart 1: line chart of copper price over time
chart1 = alt.Chart(copper_df).mark_line().encode(
    x=alt.X(
        'date:T',
        title='',
        axis=alt.Axis(grid=False)),
    y=alt.Y(
        'price:Q',
        title='$ per pound',
        axis=alt.Axis(
            grid=False,
            labelExpr='"$" + datum.value')),
    tooltip=[
        alt.Tooltip('date:T', title='Date', format='%b %Y'),
        alt.Tooltip('price:Q', title='Price ($ per pound)', format='.2f')]
)

chart1

alt.Chart(...)

In [5]:
# Load chart 2 data as df
poverty_df = pd.read_excel('Chartbook_Peru_Critical Minerals_LMC&TC_v1.xlsx', sheet_name='Chart 2', usecols=[0,1,2])
poverty_df.columns = ['year', 'peru', 'lac']
poverty_df

,year,peru,lac
0,2004,59.759946,47.25
1,2005,61.503981,44.71
2,2006,55.861864,41.02
3,2007,50.303049,38.92
4,2008,46.402334,37.00
5,2009,44.554036,36.66
6,2010,40.525380,34.72
7,2011,38.069461,32.79
8,2012,35.295254,31.63
9,2013,34.510649,30.63


In [6]:
# Create chart 2: bar chart of lac poverty headcount, with peru overlay
base = alt.Chart(poverty_df).encode(
    x=alt.X(
        'year:O',
        title='',
        axis=alt.Axis(grid=False, labelAngle=-45))
)
lac_bars = base.mark_bar().encode(
    y=alt.Y(
        'lac:Q',
        title='% of population',
        axis=alt.Axis(grid=False)),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('lac:Q', title='LAC (% of population)', format='.2f')]
)
peru_line = base.mark_line().encode(
    y=alt.Y('peru:Q'),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('peru:Q', title='Peru (% of population)', format='.2f')]
)

# Get last year's values for label positioning
last_year = poverty_df['year'].max()
last_row = poverty_df[poverty_df['year'] == last_year]
peru_label = alt.Chart(last_row).mark_text(
    align='left', dx=13, color='#E6224B'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('peru:Q'),
    text=alt.value('Peru')
)
lac_label = alt.Chart(last_row).mark_text(
    align='left', dx=13, color='#179FDB'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('lac:Q'),
    text=alt.value('LAC')
)
chart2 = lac_bars + peru_line + peru_label + lac_label

chart2

alt.LayerChart(...)

In [7]:
# Load chart 3 data as df
gdp_copper_df = pd.read_excel('Chartbook_Peru_Critical Minerals_LMC&TC_v1.xlsx', sheet_name='Chart 3', usecols=[2,3,4])
gdp_copper_df.columns = ['year', 'gdp', 'copper_price']
gdp_copper_df = gdp_copper_df[gdp_copper_df['year']>2001]
gdp_copper_df

,year,gdp,copper_price
2,2002.0,5.454,0.707416
3,2003.0,4.165,0.807017
4,2004.0,4.959,1.299997
5,2005.0,6.285,1.668693
6,2006.0,7.529,3.049089
7,2007.0,8.518,3.228750
8,2008.0,9.127,3.155192
9,2009.0,1.096,2.335966
10,2010.0,8.330,3.417671
11,2011.0,6.328,4.004394


In [8]:
# Create chart 3: bar chart of peru gdp growth, with copper price overlay
base = alt.Chart(gdp_copper_df).encode(
    x=alt.X(
        'year:O',
        title='',
        axis=alt.Axis(grid=False, labelAngle=-45))
)
gdp_bars = base.mark_bar().encode(
    y=alt.Y(
        'gdp:Q',
        title='GDP growth (annual % change)',
        scale=alt.Scale(domain=[-2,10]),
        axis=alt.Axis(
            grid=False,
            titleColor='#179FDB',
            labelColor='#179FDB',
            tickColor='#179FDB',
            values=[-2,0,2,4,6,8,10])),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('gdp:Q', title='GDP growth (% change)', format='.2f')]
)
copper_line = base.mark_line().encode(
    y=alt.Y(
        'copper_price:Q',
        title='Copper price ($ per pound)',
        scale=alt.Scale(domain=[0,6]),
        axis=alt.Axis(
            grid=False, 
            labelExpr='"$" + datum.value',
            titleColor='#E6224B',
            labelColor='#E6224B',
            tickColor='#E6224B',
            values=[0,1,2,3,4,5,6])),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('copper_price:Q', title='Copper price ($ per pound)', format='.2f')]
)
zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
    strokeWidth=1
).encode(
    y=alt.Y(
        'y:Q',
        scale=alt.Scale(domain=[-2,10]),axis=None))
chart3 = alt.layer(gdp_bars, zero_line, copper_line).resolve_scale(y='independent')

chart3

alt.LayerChart(...)

In [9]:
# Load chart 4 data
invest_df = pd.read_excel('Chartbook_Peru_Critical Minerals_LMC&TC_v1.xlsx', sheet_name='Chart 4', usecols=[1,5,6], skiprows=1, nrows=11)
invest_df.columns = ['year', 'jurisdictions', 'pct_below']
invest_df

,year,jurisdictions,pct_below
0,2015,109,0.67
1,2016,104,0.73
2,2017,91,0.79
3,2018,83,0.83
4,2019,76,0.68
5,2020,77,0.56
6,2021,84,0.50
7,2022,62,0.45
8,2023,86,0.31
9,2024,82,0.51


In [10]:
# Create chart 4: Stacked area chart of % of jurisdictions below peru
base = alt.Chart(invest_df).encode(
    x=alt.X('year:O', title='', axis=alt.Axis(grid=False))
)
area = base.mark_area(color='#E6224B', opacity=0.25).encode(
    y=alt.Y(
        'pct_below:Q', 
        title='% of jurisdictions below Peru in the global ranking',
        scale=alt.Scale(domain=[0,1]), 
        axis=alt.Axis(grid=False, labelExpr='datum.value * 100 + "%"'))
).properties(width=alt.Step(40))
line = base.mark_line(color='#E6224B').encode(
    y=alt.Y('pct_below:Q')
)
labels = base.mark_text(
    dy=25, align='center', fontWeight='bold', color='#E6224B'
).encode(
    y=alt.Y('pct_below:Q'),
    text=alt.Text('pct_below:Q', format='.0%')
).properties(width=alt.Step(40))
chart4 = area + line + labels
jurisdiction_panel = alt.Chart(invest_df).mark_text(dx=-6).encode(
    x=alt.X('year:O', 
            title='',
            axis=alt.Axis(grid=False, labels=False, ticks=False, domain=False)),
    text=alt.Text('jurisdictions:Q', format='.0f')
).properties(width=alt.Step(40), height=20)
jurisdiction_label = alt.Chart(pd.DataFrame({'text': ['Number of jurisdictions']})).mark_text(align='right', dx=-8).encode(x=alt.value(0), text='text:N').properties(width=alt.Step(40), height=20)
bottom_row = alt.layer(jurisdiction_label, jurisdiction_panel)
chart4 = alt.vconcat(chart4, bottom_row)

chart4

alt.VConcatChart(...)

In [11]:
# Load chart 5 data
gold_df = pd.read_excel('Chartbook_Peru_Critical Minerals_LMC&TC_v1.xlsx', sheet_name='Chart 5', usecols=[1,2,3], skiprows=2, nrows=21)
gold_df.columns = ['year', 'legal', 'illegal']
gold_df

,year,legal,illegal
0,2005,185.970368,27.891318
1,2006,181.234149,26.342115
2,2007,148.896961,36.709889
3,2008,158.026023,41.586405
4,2009,161.525995,55.333732
5,2010,140.423977,56.602799
6,2011,138.950792,62.980900
7,2012,137.977484,61.929554
8,2013,124.215432,63.881837
9,2014,118.368248,47.210176


In [12]:
# Create chart 5: Overlaid line charts of peru gold exports
base = alt.Chart(gold_df).encode(
    x=alt.X(
        'year:O',
        title='',
        axis=alt.Axis(grid=False, labelAngle=-45))
)
legal_line = base.mark_line(color='#179FDB').encode(
    y=alt.Y(
        'legal:Q',
        title='Tonnes',
        axis=alt.Axis(grid=False)),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('legal:Q', title='Legal exports (tonnes)', format='.2f')]
)
illegal_line = base.mark_line(color='#E6224B').encode(
    y=alt.Y('illegal:Q'),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('illegal:Q', title='Illegal exports (tonnes)', format='.2f')]
)

# First and last year's values, for both value labels and series labels
first_year = gold_df['year'].min()
last_year = gold_df['year'].max()
first_row = gold_df[gold_df['year'] == first_year]
last_row = gold_df[gold_df['year'] == last_year]
# Start value labels (positioned above/left of first point)
legal_start_value = alt.Chart(first_row).mark_text(
    align='right', dx=20, dy=-8, color='#179FDB'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('legal:Q'),
    text=alt.Text('legal:Q', format='.1f')
)
illegal_start_value = alt.Chart(first_row).mark_text(
    align='right', dx=15, dy=10, color='#E6224B'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('illegal:Q'),
    text=alt.Text('illegal:Q', format='.1f')
)
# End value labels (positioned above/left of last point)
legal_end_value = alt.Chart(last_row).mark_text(
    align='right', dx=30, dy=-8, color='#179FDB'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('legal:Q'),
    text=alt.Text('legal:Q', format='.1f')
)
illegal_end_value = alt.Chart(last_row).mark_text(
    align='right', dx=30, dy=8, color='#E6224B'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('illegal:Q'),
    text=alt.Text('illegal:Q', format='.1f')
)
# Series-name labels moved to middle year
mid_year = gold_df['year'].iloc[len(gold_df) // 2]
mid_row = gold_df[gold_df['year'] == mid_year]
legal_label = alt.Chart(mid_row).mark_text(
    align='center', dy=-20, color='#179FDB'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('legal:Q'),
    text=alt.value('Gold of legal origin')
)
illegal_label = alt.Chart(mid_row).mark_text(
    align='center', dy=25, color='#E6224B'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('illegal:Q'),
    text=alt.value('Gold of illegal origin')
)

chart5 = alt.layer(
    legal_line, illegal_line,
    legal_start_value, illegal_start_value,
    legal_end_value, illegal_end_value,
    legal_label, illegal_label
)

chart5

alt.LayerChart(...)

In [ ]:
# Save charts
styles.save(chart1, 'visualisation', 'chart1', width=450, height=360)
styles.save(chart2, 'visualisation', 'chart2', width=450, height=360)
styles.save(chart3, 'visualisation', 'chart3', width=450, height=360)
styles.save(chart4, 'visualisation', 'chart4', width=450, height=360)
styles.save(chart5, 'visualisation', 'chart5', width=450, height=360)